# Eve-3-SABER-1B: Ablation Study

**5-run ablation to validate each novel component before committing to full training.**

| Run | Config | Components Enabled |
|-----|--------|--------------------|
| 1 | baseline | Standard transformer only |
| 2 | +anchors | Slip-Anchors only |
| 3 | +experience | Experience Stream only |
| 4 | +resonant | Resonant FFN only |
| 5 | full | All three enabled |

**Token budget:** 1B tokens per run (1% of full 100B training budget). Enough to see meaningful loss separation.

**Hardware:** RunPod 1-4x A100/H100. Single GPU works fine.

**Time estimate:** ~30-60 min per run on 1x A100-80GB, ~2.5-5 hours total for all 5.

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers>=4.40.0", "accelerate>=0.28.0",
    "datasets>=2.19.0", "safetensors>=0.4.2", "tokenizers>=0.15.0",
    "wandb>=0.17.0", "matplotlib>=3.7.0", "numpy>=1.24.0", "tqdm",
    "zstandard"])

# FlashAttention 2 (optional, Ampere+ GPUs)
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "flash-attn", "--no-build-isolation"])
    print("FlashAttention 2 installed")
except Exception:
    print("flash-attn not installed (optional)")

In [ ]:
# ============================================================
# CELL 2: IMPORTS & HARDWARE DETECTION
# ============================================================
import os, sys, json, math, time, random, logging
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from accelerate import Accelerator

import numpy as np
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("ablation")

# ---- Hardware ----
if torch.cuda.is_available():
    N_GPUS = torch.cuda.device_count()
    for i in range(N_GPUS):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f"  GPU {i}: {name} ({vram:.0f} GB)")
    DEVICE = "cuda"
    DTYPE = torch.bfloat16
else:
    N_GPUS = 0
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("WARNING: No GPU detected")

print(f"\nDevice: {DEVICE}, GPUs: {N_GPUS}")

In [ ]:
# ============================================================
# CELL 3: LOAD MODEL & TOKENIZER
# ============================================================
from transformers import AutoTokenizer
from configuration_saber import SABERConfig
from modeling_saber import SABERForCausalLM

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
EOS_ID = tokenizer.eos_token_id

print(f"Tokenizer: vocab_size={tokenizer.vocab_size}, eos_id={EOS_ID}")

# ---- Ablation specs ----
ABLATION_SPECS = {
    "baseline":   {"enable_anchors": False, "enable_experience": False, "resonant_layers": []},
    "anchors":    {"enable_anchors": True,  "enable_experience": False, "resonant_layers": []},
    "experience": {"enable_anchors": False, "enable_experience": True,  "resonant_layers": []},
    "resonant":   {"enable_anchors": False, "enable_experience": False, "resonant_layers": None},  # None = default even layers
    "full":       {"enable_anchors": True,  "enable_experience": True,  "resonant_layers": None},
}

def build_model(spec_name: str) -> SABERForCausalLM:
    """Build a fresh model with the given ablation config."""
    spec = ABLATION_SPECS[spec_name]
    config = SABERConfig(
        d_ff=2855,
        enable_anchors=spec["enable_anchors"],
        enable_experience=spec["enable_experience"],
        resonant_layers=spec["resonant_layers"],
    )
    model = SABERForCausalLM(config)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"[{spec_name}] Built model: {n_params:,} params")
    return model

# Quick sanity check
_test = build_model("full")
del _test
torch.cuda.empty_cache() if DEVICE == "cuda" else None

In [ ]:
# ============================================================
# CELL 4: DATASET - STREAMING WITH SEQUENCE PACKING
# ============================================================
from datasets import load_dataset

def tokenise_example(example):
    """Extract text and tokenise. Returns token ids + EOS."""
    text = example.get("text") or example.get("content") or ""
    if not isinstance(text, str) or len(text.strip()) < 50:
        return None
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    return ids + [EOS_ID]


class PackedDataset(IterableDataset):
    """Packs tokenised documents into fixed-length sequences (no padding)."""

    def __init__(self, hf_datasets_weighted, seq_len=2048, seed=42, max_tokens=None):
        self.datasets = hf_datasets_weighted
        self.seq_len = seq_len
        self.seed = seed
        self.max_tokens = max_tokens

    def _interleave_streams(self):
        rng = random.Random(self.seed)
        iterators = [iter(ds) for ds, _ in self.datasets]
        weights = [w for _, w in self.datasets]
        while True:
            idx = rng.choices(range(len(iterators)), weights=weights, k=1)[0]
            try:
                yield next(iterators[idx])
            except StopIteration:
                iterators[idx] = iter(self.datasets[idx][0])
                yield next(iterators[idx])

    def __iter__(self):
        buffer = []
        tokens_emitted = 0
        for example in self._interleave_streams():
            ids = tokenise_example(example)
            if ids is None:
                continue
            buffer.extend(ids)
            while len(buffer) >= self.seq_len + 1:
                chunk = buffer[:self.seq_len + 1]
                buffer = buffer[self.seq_len:]
                tokens_emitted += self.seq_len
                if self.max_tokens and tokens_emitted > self.max_tokens:
                    return
                yield {
                    "input_ids": torch.tensor(chunk[:-1], dtype=torch.long),
                    "labels":    torch.tensor(chunk[1:],  dtype=torch.long),
                }


def build_ablation_dataloader(batch_size=4, max_tokens=1_000_000_000):
    """Build a streaming dataloader for ablation (FineWeb-Edu + DCLM)."""
    ds_fineweb = load_dataset(
        "HuggingFaceFW/fineweb-edu", "sample-10BT",
        split="train", streaming=True
    )
    ds_dclm = load_dataset(
        "mlfoundations/dclm-baseline-1.0", "default",
        split="train", streaming=True
    )
    packed = PackedDataset(
        hf_datasets_weighted=[(ds_fineweb, 0.6), (ds_dclm, 0.4)],
        seq_len=2048,
        max_tokens=max_tokens,
    )
    return DataLoader(packed, batch_size=batch_size, num_workers=2, pin_memory=True)

print("Dataset utilities ready.")

In [ ]:
# ============================================================
# CELL 5: ABLATION TRAINING LOOP
# ============================================================

ABLATION_TOKENS = 1_000_000_000   # 1B tokens per run
SEQ_LEN         = 2048
MICRO_BATCH     = 4               # Adjust based on your GPU VRAM
GRAD_ACCUM      = 8               # effective batch = MICRO_BATCH * GRAD_ACCUM * SEQ_LEN
LR              = 3e-4
WARMUP_STEPS    = 500
RESULTS_DIR     = Path("./ablation_results")
RESULTS_DIR.mkdir(exist_ok=True)

# Set WANDB_ENABLED = True and configure project to log to W&B
WANDB_ENABLED = False
# os.environ["WANDB_PROJECT"] = "eve-3-saber-ablation"


def run_single_ablation(spec_name: str) -> dict:
    """Train one ablation config and return metrics."""
    print(f"\n{'='*60}")
    print(f"  ABLATION: {spec_name.upper()}")
    print(f"{'='*60}\n")

    t0 = time.time()

    # ---- Build model ----
    model = build_model(spec_name)
    model = model.to(DEVICE)
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    # ---- Optimizer ----
    decay = [p for n, p in model.named_parameters() if p.dim() >= 2]
    no_decay = [p for n, p in model.named_parameters() if p.dim() < 2]
    optimizer = torch.optim.AdamW([
        {"params": decay, "weight_decay": 0.1},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=LR, betas=(0.9, 0.95), fused=(DEVICE == "cuda"))

    # ---- LR schedule (cosine with warmup) ----
    total_steps = ABLATION_TOKENS // (MICRO_BATCH * GRAD_ACCUM * SEQ_LEN)

    def lr_lambda(step):
        if step < WARMUP_STEPS:
            return step / max(1, WARMUP_STEPS)
        progress = (step - WARMUP_STEPS) / max(1, total_steps - WARMUP_STEPS)
        return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    # ---- Dataloader ----
    dataloader = build_ablation_dataloader(batch_size=MICRO_BATCH, max_tokens=ABLATION_TOKENS)

    # ---- W&B ----
    wandb_run = None
    if WANDB_ENABLED:
        import wandb
        wandb_run = wandb.init(
            project=os.environ.get("WANDB_PROJECT", "eve-3-saber-ablation"),
            name=f"ablation-{spec_name}",
            tags=["ablation", spec_name],
            reinit=True,
        )

    # ---- Train ----
    model.train()
    step = 0
    tokens_seen = 0
    loss_sum = 0.0
    loss_count = 0
    loss_curve = []
    best_loss = float("inf")
    log_interval = max(1, total_steps // 100)
    optimizer.zero_grad()

    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    pbar = tqdm(total=total_steps, desc=spec_name, unit="step")

    for batch in dataloader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        # Forward
        if DEVICE == "cuda":
            with torch.autocast(device_type="cuda", dtype=DTYPE):
                out = model(input_ids=input_ids, labels=labels)
        else:
            out = model(input_ids=input_ids, labels=labels)

        loss = out.loss / GRAD_ACCUM

        # Backward
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        # Optimizer step
        if (step + 1) % GRAD_ACCUM == 0:
            if scaler:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        raw_loss = loss.item() * GRAD_ACCUM
        loss_sum += raw_loss
        loss_count += 1
        tokens_seen += MICRO_BATCH * SEQ_LEN
        loss_curve.append(raw_loss)
        best_loss = min(best_loss, raw_loss)

        # Log
        if step % log_interval == 0 and loss_count > 0:
            avg = loss_sum / loss_count
            ppl = math.exp(min(avg, 20))
            pbar.set_postfix(loss=f"{avg:.3f}", ppl=f"{ppl:.1f}", lr=f"{scheduler.get_last_lr()[0]:.1e}")
            if wandb_run:
                wandb_run.log({
                    f"{spec_name}/loss": avg,
                    f"{spec_name}/ppl": ppl,
                    f"{spec_name}/lr": scheduler.get_last_lr()[0],
                }, step=step)
            loss_sum = 0.0
            loss_count = 0

        pbar.update(1)
        step += 1
        if step >= total_steps:
            break

    pbar.close()

    # ---- Final metrics ----
    tail = max(1, len(loss_curve) // 20)
    final_loss = sum(loss_curve[-tail:]) / tail if loss_curve else float("nan")
    final_ppl = math.exp(min(final_loss, 20))
    wall_time = time.time() - t0

    result = {
        "name": spec_name,
        "n_params": sum(p.numel() for p in model.parameters()),
        "tokens_seen": tokens_seen,
        "final_loss": final_loss,
        "final_ppl": final_ppl,
        "best_loss": best_loss,
        "wall_time_s": wall_time,
        "loss_curve_sampled": loss_curve[::max(1, len(loss_curve)//200)],
    }

    # Save
    result_path = RESULTS_DIR / f"{spec_name}_results.json"
    with open(result_path, "w") as f:
        json.dump({k: v for k, v in result.items() if k != "loss_curve_sampled"}, f, indent=2)
    print(f"\n[{spec_name}] final_loss={final_loss:.4f}  ppl={final_ppl:.2f}  time={wall_time:.0f}s")

    if wandb_run:
        wandb_run.finish()

    # Free memory
    del model, optimizer, scheduler
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

    return result

print(f"Training config: {ABLATION_TOKENS/1e9:.0f}B tokens, micro_batch={MICRO_BATCH}, grad_accum={GRAD_ACCUM}")
print(f"Effective batch: {MICRO_BATCH * GRAD_ACCUM * SEQ_LEN:,} tokens/step")
print(f"Total steps per run: {ABLATION_TOKENS // (MICRO_BATCH * GRAD_ACCUM * SEQ_LEN):,}")

In [ ]:
# ============================================================
# CELL 6: RUN ALL ABLATIONS
# ============================================================

all_results = []

for spec_name in ["baseline", "anchors", "experience", "resonant", "full"]:
    result = run_single_ablation(spec_name)
    all_results.append(result)

# Save combined results
with open(RESULTS_DIR / "ablation_summary.json", "w") as f:
    json.dump([{k: v for k, v in r.items() if k != "loss_curve_sampled"} for r in all_results], f, indent=2)

print("\nAll ablations complete!")

In [ ]:
# ============================================================
# CELL 7: RESULTS COMPARISON & VISUALIZATION
# ============================================================
import matplotlib.pyplot as plt
import json
from pathlib import Path

# ---- Load results ----
results_dir = Path("./ablation_results")
summary_path = results_dir / "ablation_summary.json"

if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
else:
    # Use in-memory results from Cell 6
    summary = [{k: v for k, v in r.items() if k != "loss_curve_sampled"} for r in all_results]

# ---- Comparison table ----
print("=" * 72)
print(f"  {'Config':<14} {'Params':>12} {'Final Loss':>12} {'Final PPL':>10} {'Time (s)':>10}")
print("-" * 72)
for r in summary:
    print(f"  {r['name']:<14} {r['n_params']:>12,} {r['final_loss']:>12.4f} {r['final_ppl']:>10.2f} {r['wall_time_s']:>10.0f}")
print("=" * 72)

# ---- Delta table (vs baseline) ----
baseline = next(r for r in summary if r["name"] == "baseline")
print(f"\n  Loss delta vs baseline ({baseline['final_loss']:.4f}):")
print(f"  {'-'*50}")
for r in summary:
    if r["name"] == "baseline":
        continue
    delta = r["final_loss"] - baseline["final_loss"]
    pct = (delta / baseline["final_loss"]) * 100
    marker = "better" if delta < 0 else "worse"
    print(f"  {r['name']:<14} {delta:>+.4f}  ({pct:>+.2f}%)  [{marker}]")

# ---- Loss curves ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {
    "baseline": "#888888",
    "anchors": "#e74c3c",
    "experience": "#3498db",
    "resonant": "#2ecc71",
    "full": "#9b59b6",
}

# Plot raw loss curves (smoothed)
for r in all_results:
    curve = r["loss_curve_sampled"]
    if len(curve) < 2:
        continue
    # Exponential moving average for smoothing
    alpha = 0.02
    smoothed = [curve[0]]
    for v in curve[1:]:
        smoothed.append(alpha * v + (1 - alpha) * smoothed[-1])
    ax1.plot(smoothed, label=r["name"], color=colors.get(r["name"], "gray"), linewidth=1.5)

ax1.set_xlabel("Step (sampled)")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss Curves")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar chart of final loss
names = [r["name"] for r in summary]
losses = [r["final_loss"] for r in summary]
bars = ax2.bar(names, losses, color=[colors.get(n, "gray") for n in names], edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Final Loss")
ax2.set_title("Final Loss by Configuration")
ax2.grid(True, alpha=0.3, axis="y")

# Annotate bars
for bar, loss in zip(bars, losses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{loss:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(results_dir / "ablation_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nPlot saved to {results_dir / 'ablation_comparison.png'}")